In [7]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("//Users/khutsosefolosha/Desktop/machine_service_risk.csv")

print("\nFirst five rows:")
print(df.head())

print("\nDataset information:")
df.info()

print("\nMissing values per column:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nTarget distribution:")
print(df["Major_Service_Required"].value_counts())

df = df.drop_duplicates()

df = df.dropna(subset=["Major_Service_Required"])

y = df["Major_Service_Required"].astype(int)

X = df.drop(
    columns=["Major_Service_Required", "Asset_ID"],
    errors="ignore"
)

numerical_cols = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_cols = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

if numerical_cols:
    numerical_imputer = SimpleImputer(strategy="median")
    X[numerical_cols] = numerical_imputer.fit_transform(
        X[numerical_cols]
    )

mode_value = X["Maintenance_Plan"].mode()[0]

X["Maintenance_Plan"] = X["Maintenance_Plan"].fillna(
    mode_value
)


if categorical_cols:
    categorical_imputer = SimpleImputer(
        strategy="most_frequent"
    )

    X[categorical_cols] = categorical_imputer.fit_transform(
        X[categorical_cols]
    )

X = pd.get_dummies(
    X,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

scaler = StandardScaler()

X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)

print("Number of records:", X_scaled.shape[0])
print("Number of predictor features:", X_scaled.shape[1])
print(
    "Missing values after preprocessing:",
    X_scaled.isnull().sum().sum()
)



First five rows:
   Asset_ID  Equipment_Age_Months Service_Contract Operating_Zone  \
0  MX-10001                    73         Extended        Central   
1  MX-10002                    89         Extended        Central   
2  MX-10003                    73         Extended         Remote   
3  MX-10004                    74         Standard        Central   
4  MX-10005                    89         Extended        Central   

  Maintenance_Plan  Average_Load_Tonnes Dust_Exposure_Level  \
0         Enhanced            22.927749                 Low   
1              NaN            26.827681                 Low   
2            Basic            17.795882                 Low   
3            Basic            33.800817                High   
4              NaN            20.716974                 Low   

   Weekly_Idle_Hours  Daily_Operating_Hours  Lubrication_Quality_Score  ...  \
0          13.297218               6.327112                   1.347214  ...   
1           4.542524          

In [8]:
kbest = SelectKBest(
    score_func=f_classif,
    k=8
)

kbest.fit(X_scaled, y)
kbest_mask = kbest.get_support()

kbest_results = pd.DataFrame({
    "Feature": X_scaled.columns[kbest_mask],
    "F_Score": kbest.scores_[kbest_mask]
})

kbest_results = kbest_results.sort_values(
    by="F_Score",
    ascending=False
).reset_index(drop=True)

print("\nTop eight predictors selected by SelectKBest:")
print(kbest_results.to_string(index=False))

selected_features_kbest = list(kbest_results["Feature"])



Top eight predictors selected by SelectKBest:
                     Feature    F_Score
  Mechanical_Condition_Score 329.785969
Maintenance_Compliance_Score 266.590232
             Error_Log_Alert 222.994948
         Sensor_Health_Score 127.915191
   Irregular_Vibration_Alert 113.792711
    Cooling_Efficiency_Score   6.887456
       Coolant_Temperature_C   3.900444
      Operating_Zone_Coastal   2.920082


In [9]:
logistic_model = LogisticRegression(
    max_iter=5000,
    solver="liblinear",
    random_state=42
)
rfe = RFE(
    estimator=logistic_model,
    n_features_to_select=8,
    step=1
)

rfe.fit(X_scaled, y)

selected_features_rfe = list(
    X_scaled.columns[rfe.support_]
)
print("\nEight predictors selected by RFE:")

for rank, feature in enumerate(selected_features_rfe, start=1):
    print(f"{rank}. {feature}")



Eight predictors selected by RFE:
1. Average_Bearing_Temperature_C
2. Coolant_Temperature_C
3. Sensor_Health_Score
4. Mechanical_Condition_Score
5. Error_Log_Alert
6. Irregular_Vibration_Alert
7. Maintenance_Compliance_Score
8. Dust_Exposure_Level_Low


In [12]:
kbest_set = set(selected_features_kbest)
rfe_set = set(selected_features_rfe)

common_features = sorted(kbest_set & rfe_set)
kbest_only = sorted(kbest_set - rfe_set)
rfe_only = sorted(rfe_set - kbest_set)

print("\nCommon predictors selected by both methods:")
for feature in common_features:
    print(feature)

print("\nPredictors selected only by SelectKBest:")
for feature in kbest_only:
    print(feature)

print("\nPredictors selected only by RFE:")
for feature in rfe_only:
    print(feature)

print("\nSummary:")
print("SelectKBest selected:", len(kbest_set), "features")
print("RFE selected:", len(rfe_set), "features")
print("Common predictors:", len(common_features))
print("SelectKBest-only predictors:", len(kbest_only))
print("RFE-only predictors:", len(rfe_only))

comparison = pd.DataFrame({
    "Common Predictors": pd.Series(common_features),
    "SelectKBest Only": pd.Series(kbest_only),
    "RFE Only": pd.Series(rfe_only)
})

print("\nFinal comparison table:")
print(comparison.to_string(index=False))

print("\nAll four sections completed successfully")


Common predictors selected by both methods:
Coolant_Temperature_C
Error_Log_Alert
Irregular_Vibration_Alert
Maintenance_Compliance_Score
Mechanical_Condition_Score
Sensor_Health_Score

Predictors selected only by SelectKBest:
Cooling_Efficiency_Score
Operating_Zone_Coastal

Predictors selected only by RFE:
Average_Bearing_Temperature_C
Dust_Exposure_Level_Low

Summary:
SelectKBest selected: 8 features
RFE selected: 8 features
Common predictors: 6
SelectKBest-only predictors: 2
RFE-only predictors: 2

Final comparison table:
           Common Predictors         SelectKBest Only                      RFE Only
       Coolant_Temperature_C Cooling_Efficiency_Score Average_Bearing_Temperature_C
             Error_Log_Alert   Operating_Zone_Coastal       Dust_Exposure_Level_Low
   Irregular_Vibration_Alert                      NaN                           NaN
Maintenance_Compliance_Score                      NaN                           NaN
  Mechanical_Condition_Score                     

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

X = X_scaled[selected_features_rfe]
y = df["Major_Service_Required"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)


logistic_model = LogisticRegression(
    max_iter=5000,
    random_state=42
)
logistic_model.fit(X_train, y_train)

tree_model = DecisionTreeClassifier(
    random_state=42
)
tree_model.fit(X_train, y_train)




,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [19]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

y_pred_logistic = logistic_model.predict(X_test)
y_pred_tree = tree_model.predict(X_test)

print("LOGISTIC REGRESSION")
print("Accuracy:", round(accuracy_score(y_test, y_pred_logistic), 4))
print("Precision:", round(precision_score(y_test, y_pred_logistic), 4))
print("Recall:", round(recall_score(y_test, y_pred_logistic), 4))
print("F1-score:", round(f1_score(y_test, y_pred_logistic), 4))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_logistic))


print("\nDECISION TREE")
print("Accuracy:", round(accuracy_score(y_test, y_pred_tree), 4))
print("Precision:", round(precision_score(y_test, y_pred_tree), 4))
print("Recall:", round(recall_score(y_test, y_pred_tree), 4))
print("F1-score:", round(f1_score(y_test, y_pred_tree), 4))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_tree))

LOGISTIC REGRESSION
Accuracy: 0.8234
Precision: 0.7487
Recall: 0.7526
F1-score: 0.7507
Confusion Matrix:
[[300  48]
 [ 47 143]]

DECISION TREE
Accuracy: 0.8978
Precision: 0.8497
Recall: 0.8632
F1-score: 0.8564
Confusion Matrix:
[[319  29]
 [ 26 164]]
